In [ ]:
INSTALL_DEPS     = True
INSTALL_BROWSERS = False
RUN_OFFLINE      = True
RUN_NETWORK      = True
RUN_BROWSERS     = False
RUN_CLI          = False

import os, sys, re, time, json, textwrap, asyncio, threading, subprocess
from pathlib import Path

WORKDIR = Path("/content/scrapling_lab") if Path("/content").exists() else Path("./scrapling_lab")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

def sh(cmd: str, quiet: bool = True) -> int:
    """Run a shell command; stream output unless quiet."""
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=quiet, text=True)
    if p.returncode and quiet:
        print((p.stderr or p.stdout or "")[-1500:])
    return p.returncode

if INSTALL_DEPS:
    sh(f'{sys.executable} -m pip install -q -U "scrapling[fetchers]" pandas')

if INSTALL_BROWSERS:
    sh("scrapling install", quiet=False)

import scrapling
from scrapling import Selector
from scrapling.fetchers import (
    Fetcher, AsyncFetcher, FetcherSession,
    DynamicFetcher,
)

W = 78
def h1(t): print("\n" + "=" * W + f"\n  {t}\n" + "=" * W)
def h2(t): print("\n--- " + t + " " + "-" * max(0, W - 5 - len(t)))
def show(label, value, n=300):
    s = str(value)
    print(f"  {label:<26} {s[:n]}{'…' if len(s) > n else ''}")

def ctext(result, default=""):
    """
    Safe '.get() then .clean()'.
    Footgun: `sel.get("")` returns a PLAIN str when it falls back to the default,
    and plain str has no .clean(), so `.get("").clean()` explodes on missing data.
    """
    v = result.get()
    return v.clean() if v is not None else default

DEMOS = []
def demo(title, enabled=True):
    """Register a demo function; failures are reported, never fatal."""
    def wrap(fn):
        DEMOS.append((title, fn, enabled))
        return fn
    return wrap

def _offload(fn, *a, **kw):
    """
    Spider.start() and asyncio.run() blow up with 'Already running asyncio in
    this thread' when a kernel already owns the event loop (JupyterLab, some
    Colab states). If a loop is live, run the call in a clean worker thread.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return fn(*a, **kw)
    box = {}
    def target():
        try:    box["v"] = fn(*a, **kw)
        except BaseException as e: box["e"] = e
    t = threading.Thread(target=target, daemon=True)
    t.start(); t.join()
    if "e" in box: raise box["e"]
    return box["v"]

def run_spider(spider):        return _offload(spider.start)
def run_async(coro_factory):   return _offload(lambda: asyncio.run(coro_factory()))

print(f"Scrapling {scrapling.__version__} | Python {sys.version.split()[0]} | cwd {os.getcwd()}")

In [ ]:
SHOP_HTML = """
<html><body>
  <div class="container">
    <h1 id="title">Gadget Store</h1>
    <section class="products" data-page="1">
      <article class="product" id="p1" data-sku="A-100">
        <h3 class="name">Alpha Phone</h3>
        <p class="price">£51.77</p><span class="stock">In stock (22)</span>
        <a class="buy" href="/item/1">Buy now</a>
      </article>
      <article class="product" id="p2" data-sku="B-200">
        <h3 class="name">  Beta   Phone </h3>
        <p class="price">£53.74</p><span class="stock">In stock (4)</span>
        <a class="buy" href="/item/2">Buy now</a>
      </article>
      <article class="product featured" id="p3" data-sku="C-300">
        <h3 class="name">Gamma Tablet</h3>
        <p class="price">£47.82</p><span class="stock">Out of stock</span>
        <a class="buy" href="/item/3">Buy now</a>
      </article>
    </section>
    <script id="payload" type="application/json">{"currency":"GBP","total":3}</script>
  </div>
</body></html>
"""

@demo("2. Parser core: Selector / Selectors / TextHandler", RUN_OFFLINE)
def part2():
    page = Selector(SHOP_HTML, url="https://shop.example.com/catalog")

    h2("CSS vs XPath — identical mental model to Scrapy/Parsel")
    show("css ::text",        page.css(".product .name::text").getall())
    show("xpath text()",      page.xpath('//p[@class="price"]/text()').getall())
    show("css ::attr(href)",  page.css("a.buy::attr(href)").getall())
    show("xpath @attr",       page.xpath("//article/@data-sku").getall())
    show(":contains() pseudo", page.css('.product:contains("Tablet") .name::text').get())

    h2("Chaining — every result is itself queryable")
    show("chained",  page.css(".products")[0].css(".product .price::text").getall())
    show("mixed",    page.xpath('//*[@class="product"]')[0].css(".name::text").get())

    h2("Selectors (the list-like container) has real superpowers")
    prods = page.css(".product")
    show("len / .length",  (len(prods), prods.length))
    show(".first / .last",  (prods.first.attrib["id"], prods.last.attrib["id"]))
    show(".re() across all", page.css(".price").re(r"[\d.]+"))
    show(".re_first()",      page.css(".price").re_first(r"[\d.]+"))
    show(".search(pred)",    prods.search(lambda e: e.has_class("featured")).attrib["id"])
    show(".filter(pred)",    [e.attrib["id"] for e in
                              prods.filter(lambda e: "Out" in e.get_all_text())])

    h2("TextHandler — a str subclass with scraping batteries included")
    messy = page.css(".product")[1].css(".name::text").get()
    show("raw text",     repr(messy))
    show(".clean()",     repr(messy.clean()))
    show(".re_first()",  page.css(".stock::text").re_first(r"\((\d+)\)"))
    show(".json()",      page.css("#payload::text").get().json())
    show("default value", page.css(".discount::text").get("N/A"))
    show("still a str",  messy.clean().upper())

    h2("One-liner record extraction")
    rows = [{
        "sku":   p.attrib["data-sku"],
        "name":  ctext(p.css(".name::text")),
        "price": float(p.css(".price").re_first(r"[\d.]+")),
        "in_stock": "In stock" in p.get_all_text(),
    } for p in page.css(".product")]
    for r in rows: show("row", r)
    return rows

@demo("3. find_all / find_by_text / find_by_regex — no selector syntax needed", RUN_OFFLINE)
def part3():
    page = Selector(SHOP_HTML, url="https://shop.example.com/catalog")

    h2("find_all(): BeautifulSoup-style, but with a waterfall filter chain")

    show("by tag",            len(page.find_all("article")))
    show("tag + class_",      len(page.find_all("article", class_="product")))
    show("attrs dict",        len(page.find_all({"class": "product"})))
    show("multiple tags",     len(page.find_all(["article", "section"])))
    show("regex on content",  len(page.find_all("p", re.compile(r"£5\d"))))
    show("predicate",         [e.attrib["id"] for e in
                               page.find_all("article", lambda e: "Out" in e.get_all_text())])
    show("find() = first",    page.find("article", class_="product").attrib["id"])

    h2("Attribute operators: $ (ends with), * (contains), ^ (starts with)")
    show("href* '/item/'",    len(page.find_all({"href*": "/item/"})))
    show("data-sku$ '00'",    [e.attrib["data-sku"] for e in page.find_all({"data-sku$": "00"})])

    h2("Text- and regex-based search (great when classes are obfuscated)")
    show("exact text",        page.find_by_text("Alpha Phone"))
    show("partial + all",     [e.text for e in
                               page.find_by_text("Phone", partial=True, first_match=False)])
    show("case sensitive",    [e.text for e in page.find_by_text(
                               "phone", partial=True, first_match=False, case_sensitive=True)])
    show("regex first",       page.find_by_regex(r"£[\d.]+").text)
    show("regex all",         [e.text for e in page.find_by_regex(r"£[\d.]+", first_match=False)])
    show("compiled regex",    page.find_by_regex(re.compile(r"Out of stock")).text)

@demo("4. DOM navigation & selector generation", RUN_OFFLINE)
def part4():
    page = Selector(SHOP_HTML, url="https://shop.example.com/catalog")

    beta = page.find_by_text("Beta Phone")

    h2("Walking the tree")
    show(".parent",           beta.parent.attrib)
    show(".next",             page.css(".product")[0].next.attrib.get("id"))
    show(".previous",         page.css(".product")[1].previous.attrib.get("id"))
    show(".siblings",         len(page.css(".product")[0].siblings))
    show(".children",         [c.tag for c in beta.parent.children])
    show(".path (ancestors)", [e.tag for e in beta.path])
    show(".iterancestors()",  [e.tag for e in beta.iterancestors()])
    show("find_ancestor()",   beta.find_ancestor(lambda e: e.has_class("products")).tag)
    show(".below_elements",   len(page.css(".products")[0].below_elements))

    h2("Element metadata")
    show(".tag / .attrib",    (beta.tag, dict(beta.parent.attrib)))
    show(".text (direct)",    repr(page.css(".product")[0].text))
    show(".get_all_text()",   repr(page.css(".product")[0].get_all_text(strip=True)[:60]))
    show(".has_class()",      page.css(".product")[2].has_class("featured"))
    show(".html_content",     page.css(".product")[0].html_content[:70])
    show(".prettify()",       page.css(".name")[0].prettify().strip())

    h2("Auto-generate reusable selectors for ANY element you found")
    show("generate_css_selector",       beta.generate_css_selector)
    show("generate_full_css_selector",  beta.generate_full_css_selector)
    show("generate_xpath_selector",     beta.generate_xpath_selector)
    show("urljoin()",                   page.urljoin("/item/9"))

In [ ]:
@demo("5. find_similar(): anchor on ONE known value, get the whole grid", RUN_OFFLINE)
def part5():
    page = Selector(SHOP_HTML, url="https://shop.example.com/catalog")

    anchor = page.find_by_text("Alpha Phone")
    card   = anchor.parent

    h2("How it works: same DOM depth -> same tag/parent/grandparent -> fuzzy attr match")
    show("similar cards found", len(card.find_similar()))
    show("ignore noisy attrs",  len(card.find_similar(ignore_attributes=["id", "data-sku"])))
    show("loosen threshold",    len(card.find_similar(similarity_threshold=0.0)))

    grid = [card] + list(card.find_similar())
    for c in grid:
        show("scraped", {"name": c.css(".name::text").get("").clean(),
                         "price": c.css(".price").re_first(r"[\d.]+")})
    print("\n  Zero brittle selectors were written for the sibling cards.")

OLD_HTML = """
<div class="container"><section class="products">
  <article class="product" id="p1"><h3>Alpha Phone</h3><p class="description">Great phone</p></article>
  <article class="product" id="p2"><h3>Beta Phone</h3><p class="description">Other phone</p></article>
</section></div>"""

NEW_HTML = """
<div class="new-container"><div class="product-wrapper"><section class="products">
  <article class="product card-v2" data-id="p1"><div class="product-info">
     <h3>Alpha Phone</h3><p class="new-description">Great phone</p></div></article>
  <article class="product card-v2" data-id="p2"><div class="product-info">
     <h3>Beta Phone</h3><p class="new-description">Other phone</p></div></article>
</section></div></div>"""

@demo("6. Adaptive scraping: the selector heals itself after a redesign", RUN_OFFLINE)
def part6():
    SITE = "https://shop.example.com/catalog"

    h2("Day 1 — scrape normally, but tag the element with auto_save=True")
    day1 = Selector(OLD_HTML, adaptive=True, url=SITE)
    el = day1.css("#p1", auto_save=True)
    show("found", el[0].css("h3::text").get())

    h2("Day 90 — the site was redesigned; ids became data-ids, wrappers appeared")
    day90 = Selector(NEW_HTML, adaptive=True, url=SITE)
    show("plain css('#p1')",  day90.css("#p1"))
    healed = day90.css("#p1", adaptive=True)
    show("adaptive=True",     healed[0].css("h3::text").get())
    show("what it matched",   dict(healed[0].attrib))

    h2("Manual mode — works with ANY selection method, not just css/xpath")
    day1.save(day1.find_by_text("Alpha Phone"), "product_title")
    props = day90.retrieve("product_title")
    show("retrieve()",  bool(props))
    show("relocate()",  day90.relocate(props, selector_type=True).css("::text").getall())

    h2("With fetchers, flip it on globally")
    print(textwrap.dedent("""
      Fetcher.configure(adaptive=True, adaptive_domain='mysite.com')
      page = Fetcher.get(url); page.css('.price', auto_save=True)   # first run
      page.css('.price', adaptive=True)                             # after redesign

      adaptive_domain= keeps one shared bucket when the URL/domain itself moves
      (e.g. an archive.org copy vs the live site). Known limit: only the FIRST
      element of a selection is fingerprinted.
    """).strip())

In [ ]:
@demo("7. Fetcher / FetcherSession / AsyncFetcher", RUN_NETWORK)
def part7():
    h2("One-off request with a real Chrome TLS + header fingerprint")
    r = Fetcher.get("https://quotes.toscrape.com/",
                    impersonate="chrome",
                    stealthy_headers=True,
                    timeout=30, retries=3, retry_delay=1,
                    follow_redirects="safe")
    show("status / reason", (r.status, r.reason))
    show("url / encoding",  (r.url, r.encoding))
    show("body bytes",      len(r.body))
    show("headers",         list(r.headers)[:5])
    show("cookies",         dict(r.cookies))
    show("request_headers", list(r.request_headers)[:6])
    show("history",         r.history)
    show("parse right off it", r.css(".quote .text::text").getall()[:2])

    h2("FetcherSession — cookies, connection reuse, one config for many calls")
    with FetcherSession(impersonate="chrome", stealthy_headers=True, timeout=30) as s:
        p1 = s.get("https://quotes.toscrape.com/")
        p2 = s.get("https://quotes.toscrape.com/page/2/")
        show("statuses", (p1.status, p2.status))
        show("page2 first quote", p2.css(".quote .text::text").get()[:60])

    h2("AsyncFetcher — N pages concurrently (sync loop vs gather)")
    urls = [f"https://quotes.toscrape.com/page/{i}/" for i in range(1, 6)]

    t0 = time.perf_counter()
    seq = [Fetcher.get(u).css(".quote").length for u in urls]
    t_seq = time.perf_counter() - t0

    async def gather_all():
        return await asyncio.gather(*(AsyncFetcher.get(u) for u in urls))
    t0 = time.perf_counter()
    pages = run_async(gather_all)
    t_par = time.perf_counter() - t0

    show("sequential", f"{seq} in {t_seq:.2f}s")
    show("concurrent", f"{[p.css('.quote').length for p in pages]} in {t_par:.2f}s")
    show("speedup",    f"{t_seq / max(t_par, 1e-9):.1f}x")

    h2("Proxy rotation (pattern — needs real proxies to run)")
    print(textwrap.dedent("""
      from scrapling.fetchers import ProxyRotator
      rot = ProxyRotator(["http://u:p@host1:8000", "http://u:p@host2:8000"])
      with FetcherSession(proxy_rotator=rot) as s:   # cyclic by default
          for u in urls: s.get(u)                    # a new proxy per request
      # Custom strategy: ProxyRotator(proxies, strategy=lambda lst, i: (random.choice(lst), i))
    """).strip())

@demo("8. DynamicFetcher: real Chromium for JS-rendered pages", RUN_BROWSERS)
def part8():
    h2("Fetch a page whose content is written by JavaScript")
    page = DynamicFetcher.fetch(
        "https://quotes.toscrape.com/js/",
        headless=True,
        network_idle=True,
        wait_selector=".quote",
        wait_selector_state="visible",
        disable_resources=False,
        timeout=60_000,
    )
    show("status", page.status)
    show("JS-rendered quotes", page.css(".quote .text::text").getall()[:2])

    h2("page_action — drive the browser before the HTML is captured")
    def scroll_and_wait(pg):
        pg.mouse.wheel(0, 4000)
        pg.wait_for_timeout(1000)
        return pg
    page2 = DynamicFetcher.fetch("https://quotes.toscrape.com/scroll",
                                 headless=True, network_idle=True,
                                 page_action=scroll_and_wait, timeout=60_000)
    show("after scrolling", page2.css(".quote").length)

    h2("Keep one browser warm across many pages")
    print(textwrap.dedent("""
      from scrapling.fetchers import DynamicSession, AsyncDynamicSession
      with DynamicSession(headless=True, disable_resources=True) as s:
          for u in urls: s.fetch(u)               # one browser, many pages

      async with AsyncDynamicSession(max_pages=4) as s:       # tab pool
          results = await asyncio.gather(*(s.fetch(u) for u in urls))
          print(s.get_pool_stats())
    """).strip())

In [ ]:
@demo("9. Spider: pagination, extra callbacks, hooks, stats, export", RUN_NETWORK)
def part9():
    from scrapling.spiders import Spider, Request, Response

    class QuotesSpider(Spider):
        name = "quotes"
        start_urls = ["https://quotes.toscrape.com/"]
        allowed_domains = {"quotes.toscrape.com"}
        concurrent_requests = 8
        concurrent_requests_per_domain = 4
        download_delay = 0.25
        robots_txt_obey = True
        max_blocked_retries = 2
        logging_level = 20

        MAX_PAGES = 3

        async def parse(self, response: Response):
            page_no = response.meta.get("page", 1)
            for q in response.css("div.quote"):
                author_url = q.css("small.author + a::attr(href)").get()
                yield {
                    "text":   ctext(q.css("span.text::text")),
                    "author": q.css("small.author::text").get(""),
                    "tags":   q.css("a.tag::text").getall(),
                    "page":   page_no,
                }
                if author_url and page_no == 1:
                    yield response.follow(author_url, callback=self.parse_author,
                                          priority=5, meta={"from": response.url})

            nxt = response.css("li.next a::attr(href)").get()
            if nxt and page_no < self.MAX_PAGES:
                yield response.follow(nxt, callback=self.parse,
                                      meta={"page": page_no + 1})

        async def parse_author(self, response: Response):
            yield {
                "type": "author",
                "name": ctext(response.css("h3.author-title::text")),
                "born": response.css("span.author-born-date::text").get(""),
                "src":  response.meta.get("from"),
            }

        async def on_start(self, resuming: bool = False):
            print(f"  [hook] crawl starting (resuming={resuming})")

        async def on_scraped_item(self, item):
            item["scraped_at"] = int(time.time())
            return item

        async def on_error(self, request, error):
            print(f"  [hook] error on {getattr(request, 'url', '?')}: {error}")

        async def on_close(self):
            print("  [hook] crawl finished")

        def is_blocked(self, response) -> bool:
            return response.status in (403, 429) or "Access Denied" in response.body[:2000]

    result = run_spider(QuotesSpider())

    h2("CrawlResult")
    show("items",            len(result.items))
    show("completed",        result.completed)
    quotes = [i for i in result.items if i.get("type") != "author"]
    authors = [i for i in result.items if i.get("type") == "author"]
    show("sample quote",     quotes[0] if quotes else None)
    show("sample author",    authors[0] if authors else None)

    h2("Stats object")
    st = result.stats
    for k in ("requests_count", "items_scraped", "items_dropped", "failed_requests_count",
              "offsite_requests_count", "robots_disallowed_count", "blocked_requests_count",
              "response_bytes", "cache_hits", "cache_misses"):
        show(k, getattr(st, k))
    show("elapsed_seconds",   round(st.elapsed_seconds, 2))
    show("requests_per_second", round(st.requests_per_second, 2))
    show("status counts",     st.response_status_count)

    h2("Built-in export")
    result.items.to_json("quotes.json", indent=True)
    result.items.to_jsonl("quotes.jsonl")
    show("quotes.json", Path("quotes.json").read_text()[:160].replace("\n", " "))

    h2("Multi-session spiders (route requests to different engines)")
    print(textwrap.dedent("""
      from scrapling.fetchers import FetcherSession, AsyncDynamicSession
      class Hybrid(Spider):
          def configure_sessions(self, manager):
              manager.add("fast",    FetcherSession(impersonate="chrome"))
              manager.add("browser", AsyncDynamicSession(headless=True), lazy=True)
          async def parse(self, response):
              yield Request(url, sid="browser", callback=self.parse_js)    # needs JS
              yield Request(url2, sid="fast",   callback=self.parse)       # cheap page
    """).strip())
    return result

@demo("10. Rule-based crawling with CrawlSpider + LinkExtractor", RUN_NETWORK)
def part10():
    from scrapling.spiders import CrawlSpider, CrawlRule, LinkExtractor, Response

    class BooksSpider(CrawlSpider):
        name = "books"
        start_urls = ["https://books.toscrape.com/"]
        allowed_domains = {"books.toscrape.com"}
        concurrent_requests = 6
        download_delay = 0.2
        logging_level = 30

        def rules(self):
            return [
                CrawlRule(
                    LinkExtractor(
                        allow=r"catalogue/[\w-]+_\d+/index\.html",
                        deny=(r"category", r"page-\d+"),
                        restrict_css="article.product_pod",
                        deny_extensions=["jpg", "png", "css", "js"],
                        canonicalize=True, strip=True,
                    ),
                    callback=self.parse_book,
                    priority=10,
                ),
            ]

        async def parse_book(self, response: Response):
            yield {
                "title": response.css("div.product_main h1::text").get(""),
                "price": response.css("p.price_color").re_first(r"[\d.]+"),
                "stock": " ".join(t.clean() for t in response.css("p.availability::text").getall()).strip(),
                "upc":   response.css("table td::text").get(""),
                "url":   response.url,
            }

    result = run_spider(BooksSpider())
    show("books scraped", len(result.items))
    for b in list(result.items)[:3]:
        show("book", b)
    print("\n  Rules run on every response handled by the default parse(); pointing a\n"
          "  rule's callback back at self.parse would make the crawl recursive.")
    return result

In [ ]:
@demo("11. stream(), development_mode cache, pause & resume", RUN_NETWORK)
def part11():
    from scrapling.spiders import Spider, Response

    class StreamSpider(Spider):
        name = "stream"
        start_urls = ["https://quotes.toscrape.com/"]
        logging_level = 40
        development_mode = True
        development_cache_dir = "./devcache"
        MAX_PAGES = 2

        async def parse(self, response: Response):
            page_no = response.meta.get("page", 1)
            for q in response.css("div.quote"):
                yield {"author": q.css("small.author::text").get(""),
                       "text": q.css("span.text::text").get("")[:48]}
            nxt = response.css("li.next a::attr(href)").get()
            if nxt and page_no < self.MAX_PAGES:
                yield response.follow(nxt, callback=self.parse, meta={"page": page_no + 1})

    h2("Streaming: consume items the moment they are scraped")
    async def consume():
        got = []
        async for item in StreamSpider().stream():
            got.append(item)
            if len(got) <= 3:
                print(f"  -> live item #{len(got)}: {item['author']}")
        return got
    items = run_async(consume)
    show("total streamed", len(items))

    h2("Development cache: iterate on parse() without re-hitting the server")
    r1 = run_spider(StreamSpider()); r2 = run_spider(StreamSpider())
    show("run 1 (miss/hit)", (r1.stats.cache_misses, r1.stats.cache_hits))
    show("run 2 (miss/hit)", (r2.stats.cache_misses, r2.stats.cache_hits))

    h2("Pause & resume (checkpointing)")
    print(textwrap.dedent("""
      spider = MySpider(crawldir="./crawl_state", interval=300)   # checkpoint every 5 min
      result = spider.start()      # Ctrl+C -> graceful pause, state written to crawldir
      if result.paused:            # rerun the SAME crawldir to resume the frontier
          MySpider(crawldir="./crawl_state").start()
      # Also: spider.start(use_uvloop=True) for a faster event loop on Linux.
    """).strip())
    return items

@demo("12. Straight into pandas", RUN_OFFLINE)
def part12():
    import pandas as pd
    src = Path("quotes.jsonl")
    if src.exists():
        rows = [json.loads(l) for l in src.read_text().splitlines() if l.strip()]
        rows = [r for r in rows if r.get("type") != "author"]
    else:
        rows = part2()
    df = pd.DataFrame(rows)
    show("shape", df.shape)
    print(df.head(5).to_string(index=False)[:900])
    if "author" in df.columns:
        h2("Top authors"); print(df["author"].value_counts().head().to_string())
        exploded = df.explode("tags")
        h2("Top tags"); print(exploded["tags"].value_counts().head().to_string())
    elif "price" in df.columns:
        h2("Price stats"); print(df["price"].describe().to_string())
    df.to_csv("dataset.csv", index=False)
    show("written", f"{Path('dataset.csv').resolve()} ({Path('dataset.csv').stat().st_size} bytes)")
    return df

@demo("13a. CLI (needs scrapling[shell])", RUN_CLI)
def part13a():
    sh("scrapling extract get 'https://quotes.toscrape.com/' page.md "
       "--css-selector '.quote' --impersonate chrome", quiet=False)
    if Path("page.md").exists():
        show("page.md", Path("page.md").read_text()[:200].replace("\n", " "))

@demo("13b. Reference card", True)
def part13b():
    print(textwrap.dedent("""
      CHOOSING A FETCHER
        Fetcher / AsyncFetcher   plain HTML, no JS. Fastest. TLS impersonation.
        DynamicFetcher           needs JS execution. Playwright Chromium.
        Spider / CrawlSpider     many pages, concurrency, retries, resume.

      GOTCHAS THAT COST PEOPLE AN HOUR
        * `pip install scrapling` gives the PARSER ONLY. Importing
          scrapling.fetchers raises ModuleNotFoundError until you install
          "scrapling[fetchers]" AND run `scrapling install` for the browsers.
        * `below_elements` and `generate_css_selector` are PROPERTIES, not methods.
        * `.next` / `.previous`, not `.next_sibling`.
        * `.text` returns an element's DIRECT text; use `.get_all_text()` for
          everything underneath it.
        * `.get("")` returns a PLAIN str when it falls back to the default, so
          `.get("").clean()` raises AttributeError the first time a field is
          missing. Use the `ctext()` helper above (or `.get()` + a None check).
        * `find_by_text()` defaults to clean_match=True and an EXACT match: search
          for "Beta Phone", not "Beta   Phone", and pass partial=True for
          substrings. A miss returns an empty `Selectors` ([]), not None.
        * CrawlSpider.rules is a METHOD returning a list; callbacks are bound
          methods, not strings.
        * Every spider callback must be `async def` + `yield` (an async generator).
        * In notebooks that already own an event loop, `Spider().start()` raises
          "Already running asyncio in this thread" — run it in a worker thread
          (see `_offload` above).
        * `headless=False` cannot work on Colab: there is no display.
        * Adaptive fingerprints live in a local SQLite file; a fresh Colab VM
          starts with an empty database, so re-run the auto_save pass first.

      AI / MCP
        pip install "scrapling[ai]" && scrapling mcp
        Exposes get/fetch tools to Claude/Cursor, pre-extracting
        content so the model burns far fewer tokens.

      ETIQUETTE
        Obey robots.txt (`robots_txt_obey = True`), keep download_delay sane,
        identify yourself, cache during development, and read the site's ToS.
        quotes.toscrape.com and books.toscrape.com exist for practice — use them.
    """).strip())

In [1]:
if __name__ == "__main__":
    t_start = time.perf_counter()
    passed, skipped, failed = [], [], []
    for title, fn, enabled in DEMOS:
        if not enabled:
            skipped.append(title); continue
        h1(title)
        try:
            fn(); passed.append(title)
        except Exception as e:
            failed.append((title, f"{type(e).__name__}: {e}"))
            print(f"  !! {type(e).__name__}: {str(e)[:300]}")

    h1("SUMMARY")
    for t in passed:            print(f"  [ok]      {t}")
    for t in skipped:           print(f"  [skipped] {t}   (flag off)")
    for t, e in failed:         print(f"  [failed]  {t}\n              {e[:200]}")
    print(f"\n  Artifacts in {os.getcwd()}: "
          f"{sorted(p.name for p in Path('.').glob('*') if p.is_file())}")
    print(f"  Total wall time: {time.perf_counter() - t_start:.1f}s")

$ /usr/bin/python3 -m pip install -q -U "scrapling[fetchers]" pandas
Scrapling 0.4.11 | Python 3.12.13 | cwd /content/scrapling_lab

  2. Parser core: Selector / Selectors / TextHandler

--- CSS vs XPath — identical mental model to Scrapy/Parsel -------------------
  css ::text                 ['Alpha Phone', '  Beta   Phone ', 'Gamma Tablet']
  xpath text()               ['£51.77', '£53.74', '£47.82']
  css ::attr(href)           ['/item/1', '/item/2', '/item/3']
  xpath @attr                ['A-100', 'B-200', 'C-300']
  :contains() pseudo         Gamma Tablet

--- Chaining — every result is itself queryable ------------------------------
  chained                    ['£51.77', '£53.74', '£47.82']
  mixed                      Alpha Phone

--- Selectors (the list-like container) has real superpowers -----------------
  len / .length              (3, 3)
  .first / .last             ('p1', 'p3')
  .re() across all           ['51.77', '53.74', '47.82']
  .re_first()                51.77
 

[2026-07-25 17:36:39] INFO: Fetched (200) <GET https://quotes.toscrape.com/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/> (referer: https://www.google.com/)
[2026-07-25 17:36:39] INFO: Fetched (200) <GET https://quotes.toscrape.com/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/> (referer: https://www.google.com/)
[2026-07-25 17:36:39] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/2/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/page/2/> (referer: https://www.google.com/)


  status / reason            (200, '')
  url / encoding             ('https://quotes.toscrape.com/', 'utf-8')
  body bytes                 11064
  headers                    ['date', 'content-type', 'strict-transport-security', 'content-encoding']
  cookies                    {}
  request_headers            ['referer']
  history                    []
  parse right off it         ['“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', '“It is our choices, Harry, that show what we truly are, far more than our abilities.”']

--- FetcherSession — cookies, connection reuse, one config for many calls ----
  statuses                   (200, 200)
  page2 first quote          “This life is what you make it. No matter what, you're going

--- AsyncFetcher — N pages concurrently (sync loop vs gather) ----------------


[2026-07-25 17:36:40] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/1/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/page/1/> (referer: https://www.google.com/)
[2026-07-25 17:36:40] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/2/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/page/2/> (referer: https://www.google.com/)
[2026-07-25 17:36:40] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/3/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/page/3/> (referer: https://www.google.com/)
[2026-07-25 17:36:40] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/4/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://quotes.toscrape.com/page/4/> (referer: https://www.google.com/)
[2026-07-25 17:36:40] INFO: Fetched (200) <GET https://quotes.toscrape.com/page/5/> (referer: https:

  sequential                 [10, 10, 10, 10, 10] in 0.57s
  concurrent                 [10, 10, 10, 10, 10] in 0.15s
  speedup                    3.8x

--- Proxy rotation (pattern — needs real proxies to run) ---------------------
from scrapling.fetchers import ProxyRotator
rot = ProxyRotator(["http://u:p@host1:8000", "http://u:p@host2:8000"])
with FetcherSession(proxy_rotator=rot) as s:   # cyclic by default
    for u in urls: s.get(u)                    # a new proxy per request
# Custom strategy: ProxyRotator(proxies, strategy=lambda lst, i: (random.choice(lst), i))

  9. Spider: pagination, extra callbacks, hooks, stats, export
  [hook] crawl starting (resuming=False)


[2026-07-25 17:36:41]:(quotes) INFO: Fetched (404) <GET https://quotes.toscrape.com/robots.txt> (referer: https://www.google.com/)
[2026-07-25 17:36:41]:(quotes) INFO: Fetched (200) <GET https://quotes.toscrape.com/> (referer: https://www.google.com/)


  [hook] crawl finished
  !! ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

  10. Rule-based crawling with CrawlSpider + LinkExtractor
  books scraped              20
  book                       {'title': 'A Light in the Attic', 'price': '51.77', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632', 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}
  book                       {'title': 'Tipping the Velvet', 'price': '53.74', 'stock': 'In stock (20 available)  In stock', 'upc': '90fa61229261140a', 'url': 'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'}
  book                       {'title': 'Soumission', 'price': '50.10', 'stock': 'In stock (20 available)  In stock  In stock', 'upc': '6957f44c3847a760', 'url': 'https://books.toscrape.com/catalogue/soumission_998/index.html'}

  Rules run on every response handled by the default parse(); pointing a
  rule's callback back at self.parse would make the